# Ejercicio 02-01-token-openai

## Objetivo

Utilizar Google Collab para crear un código Python que ayude a explicar el concepto de token.

Para ello, haremos uso de la librería de "tiktoken" que facilita trabajar con tokens en OpenAI



## Requisitos previos

N/A

## Configuración

### Paso 1: Instalar dependencia de tiktoken

Pre-requisitos:
* Requiere utilizar la dependencia de tiktoken

Pasos a seguir:

* Instalar la dependencia de tiktoken
* Verificar la versión de instalación

In [ ]:
# Install SDK "OpenAI"
!pip install tiktoken

# Verify installation
import tiktoken

print(f"✅ tiktoken {tiktoken.__version__}")

✅ tiktoken 0.13.0


### Paso 2: Creación de Cliente de Tiktoken

In [ ]:
# Configure model
MODEL = "gpt-4o-mini"

print(f"✅ Model selected: {MODEL}")

# Get the tokeniser corresponding to a specific model in the OpenAI API
tiktoken_client = tiktoken.encoding_for_model(MODEL)

print("✅ tiktoken configured successfully")

✅ Model selected: gpt-4o-mini
✅ tiktoken configured successfully


## Helpers

Se han definido una serie de funciones de soporte para el desarrollo de la parte técnica y los ejemplos utilizados

### Paso 1: Cargar funciones de ayuda para obtener metainformación sobre los tokens



In [ ]:
import unicodedata

def get_encoding(modelo: str = "gpt-4o-mini"):
    try:
        return tiktoken.encoding_for_model(modelo)
    except KeyError:
        # Fallback útil si tu versión de tiktoken no reconoce un modelo nuevo
        return tiktoken.get_encoding("o200k_base")

def count_tokens(texto: str, modelo: str = "gpt-4o-mini") -> int:
    # Normalización segura para Unicode; no traduce ni cambia el idioma
    texto = unicodedata.normalize("NFC", texto)

    enc = get_encoding(modelo)
    return len(enc.encode(texto))

def show_tokens(texto: str, modelo: str = "gpt-4o-mini", limite: int = 50):
    enc = get_encoding(modelo)
    token_ids = enc.encode(texto)

    for i, token_id in enumerate(token_ids[:limite]):
        token_bytes = enc.decode_single_token_bytes(token_id)
        token_visible = token_bytes.decode("utf-8", errors="replace")
        print(i, token_id, repr(token_visible))

def print_token_stats(items, tiktoken_client):
    print(
        f"{'Text':25s} "
        f"{'Characters':>12s} "
        f"{'Words':>8s} "
        f"{'Tokens':>8s} "
        f"{'Tok/word':>10s} "
        f"{'Tok/char':>10s}"
    )
    print("-" * 82)

    for label, text in items:
        # Obtener tokens y número de tokens
        tokens = tiktoken_client.encode(text)
        tokens_count = len(tokens)

        # Obtener número de caracteres
        character_count = len(text)

        # Obtener número de palabras
        word_count = len(text.split())

        '''
        Obtener cúanto tokens consume cada palabra, de media.
        Es decir, cuántos tokens se necesitan, de media, para representar cada palabra del texto.

        Ayuda a entender si un texto es más caro o más denso en tokens

        Ejemplo
        - Texto: "Application modernization strategy"
        - Palabras: 3
        - Tokens: 3 - 4 (dependiendo del tokenizador, del modelo, etc.)
        - Resultado: 4 tokens / 3 palabras = 1.3 tokens por palabra
        '''
        tokens_per_word = tokens_count / word_count if word_count > 0 else 0

        '''
        Obtener cuántos tokenes necesta el modelo, de medida, para representar cada carácter de texto

        Ejemplo 1
        - Texto: "Hello"
        - Caracteres: 5
        - Tokens: 1

        - Tok/char = 1 / 5 = 0.20 (cada carácter consume 0.20 tokens)

        Ejemplo 2
        - Texto: "modernization_strategy_v2"
        - Caracteres: 24
        - Tokens: 7

        - tokens_per_character = 7 / 24 = 0.29 (más alto que el ejemplo 1 porque el texto tiene guiones
        bajos, palabras técnicas o fragmentos que el tokenizador puede dividir en más pieza)
        '''
        tokens_per_character = tokens_count / character_count if character_count > 0 else 0

        print(
            f"{label:25s} "
            f"{character_count:>12d} "
            f"{word_count:>8d} "
            f"{tokens_count:>8d} "
            f"{tokens_per_word:>9.1f}x "
            f"{tokens_per_character:>9.2f}x"
        )


print("✅ Helpers functions loaded successfully")

✅ Helpers functions loaded successfully


## Consideraciones sobre tokens

### Consideración 1: Tokenización

La descomposión en tokens dependerá del modelo y el tokenizador seleccionado

#### Ejemplo: Mostrar la descomposición tokens en base a una entrada de datos dada

In [ ]:
EXAMPLE = """"Will it be raining tomorrow also in Barcelona?"""
#EXAMPLE = "What should I consider when modernizing an application?"

# Show text
print(f"Text: {EXAMPLE}")

# Get tokens
tokens = tiktoken_client.encode(EXAMPLE)

# Show details
print(f"\nTotal: {len(tokens)} tokens\n")
print("Decomposition:")
for i, t in enumerate(tokens):
    print(f" - Token {i}: '{tiktoken_client.decode([t])}' (id: {t})")

Text: What should I consider when modernizing an application?

Total: 10 tokens

Decomposition:
 - Token 0: 'What' (id: 4827)
 - Token 1: ' should' (id: 1757)
 - Token 2: ' I' (id: 357)
 - Token 3: ' consider' (id: 3331)
 - Token 4: ' when' (id: 1261)
 - Token 5: ' modern' (id: 6809)
 - Token 6: 'izing' (id: 6396)
 - Token 7: ' an' (id: 448)
 - Token 8: ' application' (id: 5200)
 - Token 9: '?' (id: 30)


#### Ejemplo: Mostrar la descomposición tokens en base a una entrada de datos dada con Helpers

In [ ]:
# Show details
encoding = get_encoding(MODEL)
print(f"Encoding : {encoding}")
print(f"- Encoding name : {encoding.name}")

tokens = count_tokens(EXAMPLE, MODEL)
print(f"\nNúmero de tokens: {tokens}")

print("\nDecomposition with limit (50):")
show_tokens(EXAMPLE, MODEL)

Encoding : <Encoding 'o200k_base'>
- Encoding name : o200k_base

Número de tokens: 10

Decomposition with limit (50):
0 1 '"'
1 17886 'Will'
2 480 ' it'
3 413 ' be'
4 131429 ' raining'
5 22021 ' tomorrow'
6 1217 ' also'
7 306 ' in'
8 23360 ' Barcelona'
9 30 '?'


### Consideración 2: El idioma del texto

En general se tiene bastante estandarizado trabajar en inglés.
- Es un idioma que es más compacto -> Consume menos tokens

Si se tiene un documento muy largo, una versión en inglés puede ocupar menos tokens que su equivalente en español. Eso significa que podrías meter más contenido dentro de la misma ventana de contexto o pagar algo menos por procesarlo.

Pero cuidado: el ahorro no siempre compensa si hay que traducir antes.

Los modelos y tokenizadores suelen estar muy optimizados para inglés porque gran parte del entrenamiento, la documentación técnica, el código, los ejemplos y los datasets históricos están en inglés.

Por lo tanto, las respuestas en inglés "suelen" ser mejores -> Calidad de la respuesta

Dependiendo de quien tenga que mantener el código o el prompt, puede compensar que este en español

La traducción ahorra tokens pero también cuesta:
- Compensa: el contenido se va a reutilizar muchas veces, el documento es muy largo, se puede cachear, se usa para prompts técnicos recurrentes, la precisión lingüística no es crítica
- No compensa: es una consulta puntual, el texto es corto, la traducción puede perder matices, la salida final debe ser fiel al original, hay terminología legal, contractual o de negocio



**TRUCOS**
- Se puede utilizar prompting, contexto, razonamiento, etc. en inglés y pedir la respuesta en español. Esto se conoce como estrategia híbrida




#### Ejemplo: Mostrar las estadísticas de token para diferentes idiomas

In [ ]:
# Load language samples

HEADPHONES_PRODUCT_SPANISH = """
Producto: Auriculares inalámbricos

Descripción:
Auriculares inalámbricos con cancelación de ruido y batería de larga duración.

Características:
- Cancelación de ruido activa
- Hasta 30 horas de batería
- Conexión Bluetooth 5.3
- Resistencia al agua IPX4

Precio: 129,99 €

Disponibilidad: En stock

Valoración: 4.6 / 5 (1.250 opiniones)
"""

HEADPHONES_PRODUCT_ENGLISH = """
Product: Wireless headphones

Description:
Wireless headphones with noise cancellation and long-lasting battery life.

Features:
- Active noise cancellation
- Up to 30 hours of battery life
- Bluetooth 5.3 connection
- IPX4 water resistance

Price: €129.99

Availability: In stock

Rating: 4.6 / 5 (1,250 reviews)
"""

HEADPHONES_PRODUCT_GERMAN = """
Produkt: Kabellose Kopfhörer

Beschreibung:
Kabellose Kopfhörer mit Geräuschunterdrückung und langer Akkulaufzeit.

Eigenschaften:
- Aktive Geräuschunterdrückung
- Bis zu 30 Stunden Akkulaufzeit
- Bluetooth-5.3-Verbindung
- Wasserbeständigkeit nach IPX4

Preis: 129,99 €

Verfügbarkeit: Auf Lager

Bewertung: 4,6 / 5 (1.250 Bewertungen)
"""

HEADPHONES_PRODUCT_JAPANESE = """
## 1. プレーンテキスト

```text
商品: ワイヤレスヘッドホン

説明:
ノイズキャンセリング機能と長時間バッテリーを備えたワイヤレスヘッドホン。

特徴:
- アクティブノイズキャンセリング
- 最大30時間のバッテリー持続時間
- Bluetooth 5.3接続
- IPX4防水性能

価格: €129.99

在庫状況: 在庫あり

評価: 4.6 / 5（1,250件のレビュー）
"""

headphones_product_plain_text_language_samples = {
    "Spanish": HEADPHONES_PRODUCT_SPANISH,
    "English": HEADPHONES_PRODUCT_ENGLISH,
    "German": HEADPHONES_PRODUCT_GERMAN,
    "Japanese": HEADPHONES_PRODUCT_JAPANESE
}

# Show tokens stats for language
print("***** Show tokens stats for language *****\n")
print_token_stats(headphones_product_plain_text_language_samples.items(), tiktoken_client)

print("\n***** Lectura rápida *****")
print("Nota:")
print("- Hay que tener en cuenta las muestras de texto concreta utilizadas para este ejemplo")
print("- Hay que tener el tokenizador utilizado y sus estrategias")
print("\nResultados:")
print("- English es el idioma que genera menos tokens -> más eficiente en tokens")
print("- Spanish genera algunos tokens más que English, aunque ambos tienen una estructura similar")
print("- German genera más tokens porque suele incluir palabras más largas o compuestas")
print("- Japanese genera muchos tokens aunque tenga menos caracteres visibles")

print("\n***** Interpretación de las métricas *****")
print("- 'Characters' indica cuántos caracteres tiene el texto.")
print("- 'Tokens' indica cuántas piezas internas necesita el modelo para procesar el texto.")
print("- 'Tok/word' indica cuántos tokens se usan de media por palabra")
print("- 'Tok/char' indica cuántos tokens se usan de media por carácter")

print("\n***** Importante sobre Japanese *****")
print("- En japonés normalmente no hay espacios entre palabras")
print("- Por eso, text.split() no detecta correctamente el número real de palabras")
print("- Como consecuencia, la métrica 'Tok/word' puede ser engañosa para Japanese")
print("- Para comparar idiomas con escrituras diferentes, 'Tok/char' suele ser más útil que 'Tok/word'")

print("\n***** Conclusión *****")
print("- Para idiomas con espacios, como Spanish, English o German, 'Tok/word' puede ser una métrica útil.")
print("- Para idiomas sin separación clara entre palabras, como Japanese, conviene mirar más 'Tok/char'.")
print("- El número de tokens depende del idioma, del contenido y de cómo el tokenizador divide el texto.")


***** Show tokens stats for language *****

Text                        Characters    Words   Tokens   Tok/word   Tok/char
----------------------------------------------------------------------------------
Spanish                            340       48       92       1.9x      0.27x
English                            315       45       78       1.7x      0.25x
German                             338       39      104       2.7x      0.31x
Japanese                           204       26      128       4.9x      0.63x

***** Lectura rápida *****
Nota:
- Hay que tener en cuenta las muestras de texto concreta utilizadas para este ejemplo
- Hay que tener el tokenizador utilizado y sus estrategias

Resultados:
- English es el idioma que genera menos tokens -> más eficiente en tokens
- Spanish genera algunos tokens más que English, aunque ambos tienen una estructura similar
- German genera más tokens porque suele incluir palabras más largas o compuestas
- Japanese genera muchos tokens aunque 

#### Ejemplo: Mostrar la descomposición tokens por idioma con Helpers

In [ ]:
# Show details
tokens = count_tokens(HEADPHONES_PRODUCT_JAPANESE, MODEL)
print(f"\nNúmero de tokens: {tokens}")

print("\nDecomposition with limit (50):")
show_tokens(HEADPHONES_PRODUCT_JAPANESE, MODEL)


Número de tokens: 128

Decomposition with limit (50):
0 198 '\n'
1 877 '##'
2 220 ' '
3 16 '1'
4 13 '.'
5 191082 ' プレ'
6 105456 'ーン'
7 16056 'テ'
8 18368 'キ'
9 38236 'スト'
10 279 '\n\n'
11 168394 '```'
12 919 'text'
13 198 '\n'
14 26198 '商品'
15 25 ':'
16 155012 ' ワ'
17 124770 'イヤ'
18 74259 'レス'
19 101370 'ヘ'
20 72760 'ッド'
21 37748 'ホ'
22 4025 'ン'
23 279 '\n\n'
24 101127 '説明'
25 734 ':\n'
26 42562 'ノ'
27 46050 'イズ'
28 161786 'キャン'
29 21870 'セ'
30 157095 'リング'
31 34453 '機'
32 6306 '能'
33 5330 'と'
34 35081 '長'
35 31402 '時間'
36 17868 'バ'
37 6660 'ッ'
38 16056 'テ'
39 36976 'リー'
40 7277 'を'
41 62193 '備'
42 18606 'え'
43 5598 'た'
44 34022 'ワ'
45 124770 'イヤ'
46 74259 'レス'
47 101370 'ヘ'
48 72760 'ッド'
49 37748 'ホ'


### Consideración 3: El formato del texto

Un token puede ser una palabra, parte de una palabra, un signo, un salto de línea, un fragmento de código o parte de una estructura.

Por tanto, también consumen tokens: títulos, separadores, tablas, comillas, llaves, corchetes, indentación, etiquetas, nombres de campos, saltos de línea, espacios repetidos, marcadores Markdown, etc.
- El formato también cuesta
- El mejor formato para IA no es el más corto, sino el más claro con el menor ruido posible.

La optimización no siempre consiste en usar menos tokens, sino en encontrar el equilibrio entre coste, claridad y calidad de respuesta.

Existen diferentes formatos

DIFERENCIA: TEXTO LIBRE vs TEXTO ESTRUCTURADO

**Texto Libre**
- Ventajas: natural, fácil de escribir y bueno para instrucciones simples
- Inconvenientes: puede ser ambiguo, dificil de parsear y puede generar respuestas menos conistentes

**Texto Estructurado**
- Ventajas: claridad, reduce ambigüedad, ayuda a controlar la salida y puede ahorrar tokens de respuesta
- Inconvenientes: añade algo de estructura, si se abusa de campos y etiquetas puede aumentar el input, etc.

FORMATOS ESTRUCTURADOS

**Texto Estructurado** (Ya se ha visto antes)

**Markdown** (Muy equilibrado)
- Ventajas: claro, legible, flexible y no demasiado pesado
- Inconvenientes: uso adecuado de los marcadores, etc.

Nota: en texto plano o Markdown, una tabla es muy didáctica pero incluye muchos caracteres extra para formar la tabla -> Consumo tokens

**JSON** (Muy útil para salidas estructuradas o integración con sistemas)
- Ventajas: muy estructurado, fácil de validar, bueno para APIs, reduce ambigüedad en salidas automáticas
- Inconvenientes: Llaves, comillas, nombres de campos y repetición consumen tokens, más largo que Markdown, cuando se tienen listas grandes, las claves repetidas multiplican el consumo

**XML y HTML** (Más caros)
- Ventajas: estructurado, generalista e incluso visual (HTML)
- Inconvenientes: las etiquetas de apertura y cierre consumen tokens, la repetición de estas etiquetas

**YAML** (Intermedio entre Markdown y JSON)
- Ventajas: más limpio que JSON, buena estructura, menos comillas y llaves
- Inconvenientes: la identación importa, puede ser más frágil, no siempre es ideal para salidas estrictamente parseables

**CSV** (Eficiente para datos tabulares)
Si tienes muchos datos en tabla, CSV puede ser más compacto que Markdown o JSON.
- Ventajas: compacto, bueno para muchas filas, menos repetición que JSON
- Inconvenientes: menos legible para humanos, no expresa bien estructuras complejas, puede ser ambiguo si hay comas dentro de los valores

**TRUCOS**
- La repetición de etiquetas dispara los tokens
- No repetir instrucciones o nombres de campos si puedes definir el formato una vez (La estructura se explica una vez y luego se reutiliza)
- Utilizar buenos separadores: [Contexto]
- Cuidado con minificar el texto quitando espacios y haciéndolo más compacto (ejemplo: Task:analyze;ctx:repo;out:risks,max5,nointro). Un prompt demasiado comprimido puede ahorrar input, pero aumentar retries y empeorar la calidad


#### Ejemplo: Mostrar las estadísticas de token para diferentes formatos de texto

In [ ]:
# Load format samples

# *******************
#  Free-form Format
# *******************

HEADPHONES_PRODUCT_SPANISH_FREE_FORM_TEXT = """
Los auriculares inalámbricos tienen cancelación de ruido activa y una batería de larga duración de hasta 30 horas. Incorporan conexión Bluetooth 5.3, resistencia al agua IPX4 y están pensados para ofrecer una experiencia cómoda y práctica en el uso diario. Su precio es de 129,99 €, están disponibles en stock y cuentan con una valoración media de 4.6 sobre 5 basada en 1.250 opiniones.
"""

# *******************
#  Structured Format
# *******************

HEADPHONES_PRODUCT_SPANISH_PLAIN_TEXT = """
Producto: Auriculares inalámbricos

Descripción:
Auriculares inalámbricos con cancelación de ruido y batería de larga duración.

Características:
- Cancelación de ruido activa
- Hasta 30 horas de batería
- Conexión Bluetooth 5.3
- Resistencia al agua IPX4

Precio: 129,99 €

Disponibilidad: En stock

Valoración: 4.6 / 5 (1.250 opiniones)
"""

HEADPHONES_PRODUCT_SPANISH_MARKDOWN = """
# Auriculares inalámbricos

Auriculares inalámbricos con cancelación de ruido y batería de larga duración.

## Características

- Cancelación de ruido activa
- Hasta 30 horas de batería
- Conexión Bluetooth 5.3
- Resistencia al agua IPX4

| Atributo | Valor |
|---|---|
| **Precio** | 129,99 € |
| **Disponibilidad** | En stock |
| **Valoración** | 4.6 / 5 (1.250 opiniones) |
"""

HEADPHONES_PRODUCT_SPANISH_JSON = """
{
  "producto": "Auriculares inalámbricos",
  "descripcion": "Auriculares inalámbricos con cancelación de ruido y batería de larga duración.",
  "caracteristicas": [
    "Cancelación de ruido activa",
    "Hasta 30 horas de batería",
    "Conexión Bluetooth 5.3",
    "Resistencia al agua IPX4"
  ],
  "precio": 129.99,
  "moneda": "EUR",
  "disponibilidad": "En stock",
  "valoracion": {
    "puntuacion": 4.6,
    "maximo": 5,
    "opiniones": 1250
  }
}
"""

HEADPHONES_PRODUCT_SPANISH_YAML = """
producto: Auriculares inalámbricos
  descripcion: >
    Auriculares inalámbricos con cancelación de ruido y batería
    de larga duración.
  caracteristicas:
    - Cancelación de ruido activa
    - Hasta 30 horas de batería
    - Conexión Bluetooth 5.3
    - Resistencia al agua IPX4
  precio: 129.99
    moneda: EUR
  disponibilidad: En stock
  valoracion:
    puntuacion: 4.6
    maximo: 5
    opiniones: 1250
"""

HEADPHONES_PRODUCT_SPANISH_XML = """
<producto>
  <nombre>Auriculares inalámbricos</nombre>
  <descripcion>
    Auriculares inalámbricos con cancelación de ruido y batería
    de larga duración.
  </descripcion>
  <caracteristicas>
    <caracteristica>Cancelación de ruido activa</caracteristica>
    <caracteristica>Hasta 30 horas de batería</caracteristica>
    <caracteristica>Conexión Bluetooth 5.3</caracteristica>
    <caracteristica>Resistencia al agua IPX4</caracteristica>
  </caracteristicas>
  <precio moneda="EUR">129.99</precio>
  <disponibilidad>En stock</disponibilidad>
  <valoracion>
    <puntuacion>4.6</puntuacion>
    <maximo>5</maximo>
    <opiniones>1250</opiniones>
  </valoracion>
</producto>
"""

HEADPHONES_PRODUCT_SPANISH_HTML = """
<div class="producto">
  <h1>Auriculares inalámbricos</h1>
  <p class="descripcion">
    Auriculares inalámbricos con cancelación de ruido y batería
    de larga duración.
  </p>
  <h2>Características</h2>
  <ul>
    <li>Cancelación de ruido activa</li>
    <li>Hasta 30 horas de batería</li>
    <li>Conexión Bluetooth 5.3</li>
    <li>Resistencia al agua IPX4</li>
  </ul>
  <table class="info">
    <tr>
      <th>Precio</th>
      <td>129,99 €</td>
    </tr>
    <tr>
      <th>Disponibilidad</th>
      <td>En stock</td>
    </tr>
    <tr>
      <th>Valoración</th>
      <td>4.6 / 5 (1.250 opiniones)</td>
    </tr>
  </table>
</div>
"""

HEADPHONES_PRODUCT_SPANISH_CSV = """
Producto;Descripción;Características;Precio;Disponibilidad;Valoración;Opiniones
Auriculares inalámbricos;Auriculares inalámbricos con cancelación de ruido y batería de larga duración.;Cancelación de ruido activa | Hasta 30 horas de batería | Conexión Bluetooth 5.3 | Resistencia al agua IPX4;129,99 €;En stock;4.6 / 5;1250
"""

headphones_product_free_form_format_samples = {
    "Spanish Free-Form Text": HEADPHONES_PRODUCT_SPANISH_FREE_FORM_TEXT
}

headphones_product_structured_format_samples = {
    "Spanish Plain Text": HEADPHONES_PRODUCT_SPANISH_PLAIN_TEXT,
    "Spanish Markdown": HEADPHONES_PRODUCT_SPANISH_MARKDOWN,
    "Spanish JSON": HEADPHONES_PRODUCT_SPANISH_JSON,
    "Spanish YAML": HEADPHONES_PRODUCT_SPANISH_YAML,
    "Spanish XML": HEADPHONES_PRODUCT_SPANISH_XML,
    "Spanish HTML": HEADPHONES_PRODUCT_SPANISH_HTML,
    "Spanish CSV": HEADPHONES_PRODUCT_SPANISH_CSV
}

# Show tokens stats for free-form formats
print("***** Show tokens stats for free-form format *****\n")
print_token_stats(headphones_product_free_form_format_samples.items(), tiktoken_client)

# Show tokens stats for structured formats
print("\n***** Show tokens stats for structured formats *****\n")
print_token_stats(headphones_product_structured_format_samples.items(), tiktoken_client)

print("\n***** Lectura rápida *****")
print("- En esta muestra, CSV es el formato más compacto en tokens, con 88 tokens")
print("- Free-form text y Plain Text tienen un coste muy parecido: 91 y 92 tokens respectivamente")
print("- Markdown añade algo más de coste porque incorpora sintaxis adicional como títulos, listas y tablas")
print("- YAML y JSON consumen más tokens porque incluyen nombres de campos, indentación, comillas, llaves y estructura")
print("- XML y HTML son los formatos más costosos porque repiten muchas etiquetas de apertura y cierre")

print("\n***** Interpretación por formato *****")
print("- Free-form text es eficiente y fácil de leer, pero menos fiable si luego necesitamos extraer datos concretos")
print("- Plain Text estructurado mantiene buena legibilidad y un coste bajo")
print("- Markdown es útil para mostrar información al usuario, pero puede aumentar tokens por tablas y símbolos")
print("- JSON es muy útil para integraciones y procesamiento automático, aunque tiene más overhead")
print("- YAML suele ser más legible que JSON, pero también añade estructura y puede ser sensible a la indentación")
print("- XML y HTML son expresivos, pero tienen mucho overhead por las etiquetas")
print("- CSV puede ser muy compacto, pero pierde claridad cuando hay listas, textos largos o campos complejos")

print("\n***** Sobre las métricas *****")
print("- 'Tokens' es la métrica principal para estimar coste")
print("- 'Tok/word' puede subir mucho en formatos estructurados porque símbolos, etiquetas y separadores también generan tokens")
print("- 'Tok/char' ayuda a comparar cuánto overhead introduce cada formato respecto a su tamaño en caracteres")
print("- En formatos como JSON, XML o HTML, no todos los tokens representan contenido de negocio; muchos representan estructura")

print("\n***** Conclusión *****")
print("- Si el objetivo es minimizar coste, formatos compactos como CSV, free-form o plain text pueden ser más eficientes")
print("- Si el objetivo es fiabilidad, validación o integración con sistemas, JSON o YAML pueden compensar el mayor coste")
print("- Si el objetivo es presentación visual, Markdown o HTML pueden ser útiles, aunque consuman más tokens")
print("- La mejor elección depende del equilibrio entre coste, legibilidad y facilidad de procesamiento")

***** Show tokens stats for free-form format *****

Text                        Characters    Words   Tokens   Tok/word   Tok/char
----------------------------------------------------------------------------------
Spanish Free-Form Text             388       64       91       1.4x      0.23x

***** Show tokens stats for structured formats *****

Text                        Characters    Words   Tokens   Tok/word   Tok/char
----------------------------------------------------------------------------------
Spanish Plain Text                 340       48       92       1.9x      0.27x
Spanish Markdown                   378       63      114       1.8x      0.30x
Spanish JSON                       458       52      145       2.8x      0.32x
Spanish YAML                       413       51      120       2.4x      0.29x
Spanish XML                        681       44      199       4.5x      0.29x
Spanish HTML                       641       59      224       3.8x      0.35x
Spanish CSV     

#### Ejemplo: Mostrar la descomposición tokens por formato de texto con Helpers

In [ ]:
# Show details
tokens = count_tokens(HEADPHONES_PRODUCT_SPANISH_HTML, MODEL)
print(f"\nNúmero de tokens: {tokens}")

print("\nDecomposition with limit (50):")
show_tokens(HEADPHONES_PRODUCT_SPANISH_HTML, MODEL)


Número de tokens: 224

Decomposition with limit (50):
0 198 '\n'
1 5878 '<div'
2 744 ' class'
3 580 '="'
4 91050 'producto'
5 1540 '">\n'
6 220 ' '
7 464 ' <'
8 71 'h'
9 16 '1'
10 63198 '>A'
11 330 'ur'
12 125014 'iculares'
13 179592 ' inalám'
14 1697 'br'
15 6302 'icos'
16 808 '</'
17 71 'h'
18 16 '1'
19 523 '>\n'
20 220 ' '
21 464 ' <'
22 79 'p'
23 744 ' class'
24 580 '="'
25 87971 'descripcion'
26 1540 '">\n'
27 271 '   '
28 47295 ' Aur'
29 125014 'iculares'
30 179592 ' inalám'
31 1697 'br'
32 6302 'icos'
33 406 ' con'
34 13491 ' cancel'
35 2119 'ación'
36 334 ' de'
37 128173 ' ruido'
38 342 ' y'
39 100373 ' batería'
40 198 '\n'
41 271 '   '
42 334 ' de'
43 66867 ' larga'
44 99692 ' duración'
45 558 '.\n'
46 220 ' '
47 1040 ' </'
48 79 'p'
49 523 '>\n'


### Consideración 4: Trabajar con código

#### Ejemplo: Mostrar las estadísticas de token para diferentes códigos

In [ ]:
# Load code samples

CODE_PYTHON = """
def fibonacci_recursive(n: int) -> int:
    if n < 0:
        raise ValueError("n must be >= 0")

    if n <= 1:
        return n

    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)


def fibonacci_iterative(n: int) -> int:
    if n < 0:
        raise ValueError("n must be >= 0")

    a, b = 0, 1

    for _ in range(n):
        a, b = b, a + b

    return a

print(fibonacci_recursive(10))  # 55
print(fibonacci_iterative(10))  # 55
"""

CODE_JAVA = """
public class Fibonacci {

    public static long fibonacciRecursive(int n) {
        if (n < 0) {
            throw new IllegalArgumentException("n must be >= 0");
        }

        if (n <= 1) {
            return n;
        }

        return fibonacciRecursive(n - 1) + fibonacciRecursive(n - 2);
    }

    public static long fibonacciIterative(int n) {
        if (n < 0) {
            throw new IllegalArgumentException("n must be >= 0");
        }

        long a = 0;
        long b = 1;

        for (int i = 0; i < n; i++) {
            long temp = a;
            a = b;
            b = temp + b;
        }

        return a;
    }

    public static void main(String[] args) {
        System.out.println(fibonacciRecursive(10)); // 55
        System.out.println(fibonacciIterative(10)); // 55
    }
}
"""

CODE_RUST = """
fn fibonacci_recursive(n: u64) -> u64 {
    if n <= 1 {
        return n;
    }

    fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)
}

fn fibonacci_iterative(n: u64) -> u64 {
    let mut a = 0;
    let mut b = 1;

    for _ in 0..n {
        let temp = a;
        a = b;
        b = temp + b;
    }

    a
}

fn main() {
    println!("{}", fibonacci_recursive(10)); // 55
    println!("{}", fibonacci_iterative(10)); // 55
}
"""

CODE_GO = """
package main

import "fmt"

func fibonacciRecursive(n uint64) uint64 {
	if n <= 1 {
		return n
	}

	return fibonacciRecursive(n-1) + fibonacciRecursive(n-2)
}

func fibonacciIterative(n uint64) uint64 {
	var a uint64 = 0
	var b uint64 = 1

	for i := uint64(0); i < n; i++ {
		temp := a
		a = b
		b = temp + b
	}

	return a
}

func main() {
	fmt.Println(fibonacciRecursive(10)) // 55
	fmt.Println(fibonacciIterative(10)) // 55
}
"""

code_samples = {
    "Python": CODE_PYTHON,
    "Java": CODE_JAVA,
    "Rust": CODE_RUST,
    "Go": CODE_GO

}

# Show tokens stats for code language
print("***** Show tokens stats for code language *****\n")
print_token_stats(code_samples.items(), tiktoken_client)

print("\n***** Lectura rápida *****")
print("- En esta muestra, Rust es el lenguaje que genera menos tokens: 135 tokens")
print("- Go queda muy cerca de Rust, con 140 tokens")
print("- Python genera 146 tokens, aunque el código parece visualmente sencillo y compacto")
print("- Java es el que más tokens genera, con 208 tokens, principalmente porque necesita más estructura: clase, métodos estáticos, tipos explícitos y método main")

print("\n***** Interpretación de las métricas *****")
print("- 'Characters' indica el tamaño visible del código en caracteres")
print("- 'Words' cuenta fragmentos separados por espacios, pero en código esta métrica puede ser poco representativa")
print("- 'Tokens' es la métrica más importante para estimar coste de procesamiento por el modelo")
print("- 'Tok/word' puede ser engañosa en código porque símbolos como (), {}, ;, _, ==, <= o + también consumen tokens")
print("- 'Tok/char' ayuda a ver la densidad de tokens respecto al tamaño del código")

print("\n***** Nota sobre 'Words' en código *****")
print("- En lenguaje natural, una palabra suele tener significado claro")
print("- En código, text.split() solo separa por espacios, pero no entiende la sintaxis del lenguaje")
print("- Por ejemplo, fibonacci_recursive(n - 1) puede contener varios tokens aunque parezca una sola expresión")
print("- Por eso, para código es mejor fijarse en 'Tokens' y 'Tok/char' que en 'Tok/word'")

print("\n***** Observaciones por lenguaje *****")
print("- Python tiene menos estructura obligatoria que Java, pero nombres como fibonacci_recursive pueden dividirse en varios tokens")
print("- Java tiene más caracteres y más tokens porque requiere clase, tipos, modificadores, llaves y punto y coma")
print("- Rust es compacto en esta muestra y obtiene el menor número de tokens")
print("- Go también es bastante compacto, aunque incluye package, import, tipos y una función main")

print("\n***** Conclusión *****")
print("- En código, menos caracteres no siempre significa muchos menos tokens")
print("- La sintaxis del lenguaje influye directamente en el coste de tokens")
print("- Para estimar coste en casos reales, conviene medir el código con el tokenizador en lugar de asumir que todos los lenguajes cuestan igual")
print("- Si el objetivo es reducir tokens, también importan los nombres de variables, comentarios, espacios, repetición de estructuras y formato del código")




***** Show tokens stats for code language *****

Text                        Characters    Words   Tokens   Tok/word   Tok/char
----------------------------------------------------------------------------------
Python                             452       68      146       2.1x      0.32x
Java                               813      107      208       1.9x      0.26x
Rust                               435       70      135       1.9x      0.31x
Go                                 429       71      140       2.0x      0.33x

***** Lectura rápida *****
- En esta muestra, Rust es el lenguaje que genera menos tokens: 135 tokens
- Go queda muy cerca de Rust, con 140 tokens
- Python genera 146 tokens, aunque el código parece visualmente sencillo y compacto
- Java es el que más tokens genera, con 208 tokens, principalmente porque necesita más estructura: clase, métodos estáticos, tipos explícitos y método main

***** Interpretación de las métricas *****
- 'Characters' indica el tamaño visible d

#### Ejemplo: Mostrar la descomposición tokens por código con Helpers

In [ ]:
# Show details
tokens = count_tokens(CODE_PYTHON, MODEL)
print(f"\nNúmero de tokens: {tokens}")

print("\nDecomposition with limit (50):")
show_tokens(CODE_PYTHON, MODEL)


Número de tokens: 146

Decomposition with limit (50):
0 198 '\n'
1 1314 'def'
2 165916 ' fibonacci'
3 175926 '_recursive'
4 2406 '(n'
5 25 ':'
6 677 ' int'
7 8 ')'
8 2747 ' ->'
9 677 ' int'
10 734 ':\n'
11 271 '   '
12 538 ' if'
13 297 ' n'
14 464 ' <'
15 220 ' '
16 15 '0'
17 734 ':\n'
18 309 '       '
19 9338 ' raise'
20 9963 ' Value'
21 2255 'Error'
22 568 '("'
23 77 'n'
24 2804 ' must'
25 413 ' be'
26 5064 ' >='
27 220 ' '
28 15 '0'
29 10822 '")\n\n'
30 271 '   '
31 538 ' if'
32 297 ' n'
33 5017 ' <='
34 220 ' '
35 16 '1'
36 734 ':\n'
37 309 '       '
38 622 ' return'
39 297 ' n'
40 279 '\n\n'
41 271 '   '
42 622 ' return'
43 165916 ' fibonacci'
44 175926 '_recursive'
45 2406 '(n'
46 533 ' -'
47 220 ' '
48 16 '1'
49 8 ')'
